# Headless image registration: chaining Fractal tasks

This notebook aligns the imaging cycles of a multiplexed OME-Zarr plate with three [Fractal](https://fractal-analytics-platform.github.io/) tasks from the `fractal-tasks-core` package, run from Python with the API of `workflow_tasks_core`:

1. **Calculate Registration (image-based)** finds the shift between each cycle and the reference cycle.
2. **Find Registration Consensus** finds the region all cycles share.
3. **Apply Registration to Image** shifts the images and crops them to that region.

Each task works on what the previous one wrote into the OME-Zarr. After a run, `task.zarr_urls` lists the images to continue with, so it is passed straight to the next task.

The [segmentation example](segmentation.ipynb) covers the basics (catalog, installing, arguments, outputs) in more detail.

**Requirements**
- `pixi` on your `PATH` (or in `~/.pixi/bin`). Each task package runs in its own pixi environment.
- A kernel with `napari-workflow-tasks` installed, e.g. the repository's pixi environment (`pixi run python -m ipykernel install --user --name napari-workflow-tasks`).
- The example dataset `220605_151046_mip.zarr`, unzipped: one well (`C/02`) imaged in two cycles, with DAPI in both. Image `0` is cycle 0 (the reference), image `1` is cycle 1.

## Settings

In [ ]:
from pathlib import Path

PLATE = Path.home() / "Downloads" / "220605_151046_mip.zarr"
WAVELENGTH = "A04_C01"  # the DAPI channel, imaged in both cycles

The tasks write into this OME-Zarr: registration results go into its ROI tables, and the last task **replaces the image data with the aligned images**. To start over with the unaligned images, unzip the dataset again.

## Get the task package

In [ ]:
from workflow_tasks_core import Catalog

pkg = Catalog().package("fractal-tasks-core")
pkg.install()
print(pkg)
print("Tasks:", pkg.tasks)

In [ ]:
calculate = pkg.task("Calculate Registration (image-based)")
consensus = pkg.task("Find Registration Consensus")
apply_registration = pkg.task("Apply Registration to Image")

The first two are *compound* tasks: a setup step runs once on all images (here, it pairs each cycle with the reference cycle of its well), then a compute step runs on each image. `help()` lists the arguments of each step. `run()` takes them all as keyword arguments and gives each step its own.

In [ ]:
calculate.help()

## 1. Calculate the shifts

`run()` takes the whole plate: it runs the task on all images of all its wells. The images are small, so the full resolution (`level_path="0"`) is used instead of the default `"2"`.

In [ ]:
calculate.run(PLATE, wavelength_id=WAVELENGTH, level_path="0")
calculate.zarr_urls

The task added the shift of each field of view (FOV) to the image's `FOV_ROI_table` (`translation_x`, `translation_y`, `translation_z`, in micrometers). The reference cycle has no shift; cycle 1 is shifted against it:

In [ ]:
from ngio import open_ome_zarr_container

shift_columns = ["translation_x", "translation_y", "translation_z"]
cycle_1 = calculate.zarr_urls[1]
open_ome_zarr_container(cycle_1).get_table("FOV_ROI_table").dataframe[shift_columns]

### Before: the cycles in napari

`show()` opens a run's images in napari. Showing cycle 0 in green and cycle 1 in magenta makes the shift visible: where the nuclei overlap they are white.

In [ ]:
from napari.utils import nbscreenshot


def overlay(viewer):
    """Cycle 0 green, cycle 1 magenta, blended."""
    for layer in viewer.layers:
        layer.colormap = "green" if layer.name.startswith("R0") else "magenta"
        layer.blending = "additive"
    viewer.reset_view()
    return nbscreenshot(viewer)


before = calculate.show()
overlay(before)

## 2. Find the consensus region

The next task runs on the images the previous one returned. It writes the `FOV_ROI_table_registered` table: the part of each FOV that all cycles cover.

In [ ]:
consensus.run(calculate.zarr_urls)

## 3. Apply the registration

`registered_roi_table` names the table from step 2. The task shifts each cycle onto the reference, masks the area outside the consensus region, and with `overwrite_input=True` (the default) replaces the image data. It also aligns the image's label images (`register_labels=True`).

With `overwrite_input=False` the aligned images go next to the originals (`0_registered`, ...) instead. These are added to the well as further images of the same cycles, and the registration tasks then no longer run on that well.

In [ ]:
apply_registration.run(
    consensus.zarr_urls,
    registered_roi_table="FOV_ROI_table_registered",
)
apply_registration.zarr_urls

### After: the aligned cycles

`show()` adds to the current napari viewer, which still shows the unaligned images. A new viewer shows only the aligned ones:

In [ ]:
import napari

after = apply_registration.show(napari.Viewer())
overlay(after)

## The whole chain

Put together, registering a plate is three calls. Each one fails with an exception if its task fails, so a later step never runs on missing results:

```python
calculate.run(PLATE, wavelength_id=WAVELENGTH, level_path="0")
consensus.run(calculate.zarr_urls)
apply_registration.run(consensus.zarr_urls, registered_roi_table="FOV_ROI_table_registered")
```

`apply_registration.zarr_urls` then lists the aligned images for the next task, e.g. a segmentation as in the [segmentation example](segmentation.ipynb).